In [20]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Veri setini yükle
CSV_FILENAME = "../datasets/cancer_data.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (569, 33)


In [2]:
# ==========================================
# 1.1 İLK KEŞİF
# ==========================================

print("\n" + "=" * 50)
print("🔍 EKSİK VERİ ANALİZİ (Missing Values)")
print("=" * 50)

# Sadece eksik verisi olan sütunları ve oranlarını hesapla
missing_count = df.isnull().sum()
missing_percent = 100 * df.isnull().mean()

missing_df = pd.DataFrame(
    {"Eksik Sayısı": missing_count, "Oran (%)": missing_percent}
)
# Sadece eksik değeri olanları filtrele ve büyükten küçüğe sırala
missing_df = missing_df[missing_df["Eksik Sayısı"] > 0].sort_values(
    by="Eksik Sayısı", ascending=False
)

if missing_df.empty:
  print("✨ Harika! Veri setinde hiç eksik değer yok.")
else:
  print(missing_df.to_string())

print("=" * 50 + "\n")


🔍 EKSİK VERİ ANALİZİ (Missing Values)
             Eksik Sayısı  Oran (%)
Unnamed: 32           569     100.0



In [3]:
# ==========================================
# 1.2 KEŞİFSEL VERİ ANALİZİ
# ==========================================
# 1. Genel Yapı ve Tipler
print("--- Bilgiler ve Tipler ---")
print(df.info())

# 2. İstatistiksel Dağılım
print("\n--- İstatistiksel Özet ---")
display(df.describe())

# 3. Eksik Değer Kontrolü
print("\n--- Eksik Değerler ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# 4. İlk 5 Satır Göz Atma
print("\n--- İlk 5 Satır ---")
display(df.head())

--- Bilgiler ve Tipler ---
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    int64  
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,0.0
mean,3.037183e+07,0.372583,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,NaN
std,1.250206e+08,0.483918,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,NaN
min,8.670000e+03,0.000000,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,NaN
25%,8.692180e+05,0.000000,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,NaN
50%,9.060240e+05,0.000000,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,NaN
75%,8.813129e+06,1.000000,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,NaN
max,9.113205e+08,1.000000,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,NaN



--- Eksik Değerler ---
Unnamed: 32    569
dtype: int64

--- İlk 5 Satır ---


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
# ==========================================
# 1.3 VERİ TEMİZLEME VE ÖN İŞLEME
# ==========================================

# Gereksiz sütunları uçur
drop_cols = ["id", "Unnamed: 32"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [22]:
from sklearn.calibration import CalibratedClassifierCV

# ==========================================
# 1.4 ÖZNİTELİK MÜHENDİSLİĞİ
# ==========================================

# Sadece float64 tipindeki sütunları otomatik seçer
numeric_cols = (
    df.select_dtypes(include=["float64"])
    .drop(columns=["Unnamed: 32"], errors="ignore")
    .columns.tolist()
)

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "poly",
            PolynomialFeatures(
                degree=2, interaction_only=False, include_bias=False
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_cols)],
    remainder="passthrough",
)

# RBF kernel için degree parametresi kaldırıldı, proba için CalibratedClassifierCV eklendi
base_svc = SVC(
    kernel="rbf", gamma="scale", C=1, random_state=42
)

pipeline_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=20)),
        ("classifier", CalibratedClassifierCV(base_svc, ensemble=False)),
    ]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring="accuracy")

for i, score in enumerate(cv_scores, 1):
  print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("=" * 40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 0.9825
Fold 2: 0.9474
Fold 3: 0.9737
Fold 4: 0.9649
Fold 5: 0.9912

Gerçek K-Fold Başarısı (Ortalama): 0.9719
Skor Sapması (Standart Sapma)    : 0.0151



In [21]:
from sklearn.calibration import CalibratedClassifierCV

# ==========================================
# ARA BÖLÜM: İDEAL K DEĞERİNİ BULMA
# ==========================================
for k_val in [10, 15, 20, 25, 30, 35]:
  temp_pipeline = Pipeline([
      ("preprocessor", preprocessor),
      ("feature_selection", SelectKBest(score_func=f_classif, k=k_val)),
      (
          "classifier",
          CalibratedClassifierCV(
              SVC(
                  kernel="rbf",
                  gamma="scale",
                  degree=4,
                  C=1,
                  random_state=42,
              ),
              ensemble=False,
          ),
      ),
  ])

  scores = cross_val_score(temp_pipeline, X, y, cv=skf, scoring="accuracy")
  print(f"k = {k_val} için Ortalama K-Fold Başarısı: {np.mean(scores):.4f}")

k = 10 için Ortalama K-Fold Başarısı: 0.9438
k = 15 için Ortalama K-Fold Başarısı: 0.9438
k = 20 için Ortalama K-Fold Başarısı: 0.9719
k = 25 için Ortalama K-Fold Başarısı: 0.9684
k = 30 için Ortalama K-Fold Başarısı: 0.9684
k = 35 için Ortalama K-Fold Başarısı: 0.9702


In [23]:
# ==========================================
# 2. NİHAİ MODEL EĞİTİMİ VE KAYIT
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))



Train Doğruluk Oranı: 0.9846
Model Doğruluk Oranı (Accuracy): 0.9825

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99        72
           1       0.98      0.98      0.98        42

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [24]:
# ==========================================
# 3. KARA KUTUYU AÇMA: Hangi Özellikler Seçildi?
# ==========================================
preprocessor = pipeline_model.named_steps["preprocessor"]
selector = pipeline_model.named_steps["feature_selection"]
# classifier adımı non-linear olduğu için coef_ kullanılmıyor

all_feature_names = preprocessor.get_feature_names_out()
selected_mask = selector.get_support()
selected_features = all_feature_names[selected_mask]

# SelectKBest'in özellikler için hesapladığı istatistiksel skorları alıyoruz
feature_scores = selector.scores_[selected_mask]

feature_importance = pd.DataFrame(
    {"Özellik (Feature)": selected_features, "Önem Skoru": feature_scores}
)

# --- GÖRSEL TEMİZLİK VE SIRALAMA BÖLÜMÜ ---
feature_importance["Özellik (Feature)"] = feature_importance[
    "Özellik (Feature)"
].str.replace(r"^(num__|cat__|text__|remainder__)", "", regex=True)

feature_importance = feature_importance.sort_values(
    by="Önem Skoru", ascending=False
)

feature_importance["Önem Skoru"] = feature_importance["Önem Skoru"].round(4)

# ------------------------------------------

print("\n--- Modelin Seçtiği En İyi Özellikler ve Önem Skorları ---")
print(feature_importance.to_string(index=False))

print(
    "En düşük tahmin edilen olasılık:",
    pipeline_model.predict_proba(X_test)[:, 1].min(),
)


--- Modelin Seçtiği En İyi Özellikler ve Önem Skorları ---
   Özellik (Feature)  Önem Skoru
concave points_worst    733.7249
     perimeter_worst    717.2465
        radius_worst    692.8614
 concave points_mean    684.5268
      perimeter_mean    548.4132
          area_worst    522.1889
         radius_mean    511.2748
           area_mean    444.8575
      concavity_mean    397.5921
     concavity_worst    319.5078
    compactness_mean    263.5643
   compactness_worst    238.2037
           radius_se    205.4332
        perimeter_se    193.1689
             area_se    180.5612
       texture_worst    126.1191
    smoothness_worst    103.7233
      symmetry_worst    100.5594
        texture_mean     93.4815
   concave points_se     87.5061
En düşük tahmin edilen olasılık: 0.0003186898276969652


In [25]:
# ==========================================
# 4. MODELİ KAYDETME
# ==========================================
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(pipeline_model, "../backend/models/cancer_data_pipeline.pkl")
joblib.dump(list(X_train.columns), "../backend/models/model_columns_svm.pkl")

print(
    "Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne"
    " kaydedildi!"
)

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!
